# EP.03.01. Streaming
**Materia:** Manejo Masivo de Datos  
**Tecnología:** Apache Spark 3.5.3 · PySpark Structured Streaming · OpenBSD Netcat (`nc`)  
**Entorno:** Ubuntu Linux · Python 3.12 · JupyterLab  
###  Objetivo de la Práctica
Comprender e implementar el procesamiento de flujos de datos en **tiempo real** mediante **Spark Structured Streaming**, conectando un origen continuo de texto por sockets TCP (`localhost:9999`) y analizando las diferencias entre transformaciones con y sin estado, modos de salida (`complete`, `update`, `append`) y control de frecuencia mediante *triggers* (`processingTime="1 minute"`).

## 1. Inicialización de la Sesión de Spark Streaming

Para procesar datos en tiempo real de forma local configuramos `SparkSession` con al menos **2 hilos de ejecución** (`local[2]`): uno dedicado exclusivamente a escuchar y recibir los datos entrantes desde el puerto de red, y el segundo para procesar cada micro-lote (*micro-batch*).

In [2]:
import os
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

# Silenciar logs informativos de JVM/Spark para salidas limpias
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=false pyspark-shell"
logging.getLogger("py4j").setLevel(logging.ERROR)

# Crear SparkSession con 2 hilos (1 receptor + 1 procesador, mínimo para streaming)
spark = (
    SparkSession.builder
    .appName("Streaming IABD WordCount")
    .master("local[2]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Streaming listo | Versión: {spark.version} | Master: {spark.sparkContext.master}")

Spark Streaming listo | Versión: 3.5.3 | Master: local[2]


## 2. Caso 1: Hola Spark Streaming (`WordCount` en tiempo real)

Construimos el flujo completo de lectura desde el socket TCP `localhost:9999`, separación de palabras con `split()` y `explode()`, y conteo acumulado mediante `groupBy("palabra").count()` usando `.outputMode("complete")`.

In [3]:
# Caso 1: Hola Spark Streaming (WordCount en tiempo real)
lineasDF = (
    spark.readStream
    .format("socket")
    .option("host", "localhost")
    .option("port", 9999)
    .load()
)

palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

wordCountQuery = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)

wordCountQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|streaming|    1|
|    spark|    1|
|     hola|    1|
+---------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|streaming|    1|
|    spark|    2|
|     hola|    2|
|    mundo|    1|
+---------+-----+

-------------------------------------------
Batch: 3
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   nombre|    1|
|streaming|    1|
|       es|    1|
|    spark|    2|
|     hola|    2|
|       mi|    1|
| valentin|    3|
|    mundo|    1|
+---------+-----+


## 3. Ejercicio 1: Comparación de Modos de Salida (`update` y `append`)

### 3.1. Modo de salida `update`
Únicamente emite hacia la consola las filas nuevas o cuyo conteo se modificó en el micro-batch actual.

In [4]:
# Detener streams previos activos
for q in spark.streams.active:
    q.stop()

# Ejercicio 1: Comparación de modos de salida - Modo "update"
wordCountUpdate = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("update")
    .start()
)

wordCountUpdate.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 4
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   nombre|    1|
|streaming|    1|
|       es|    1|
|    spark|    3|
|     hola|    2|
|       mi|    1|
|    nuevo|    1|
| valentin|    3|
|    mundo|    1|
|     dato|    1|
+---------+-----+


### 3.2. Modo de salida `append` (Transformaciones sin estado)
Aplicado sobre una consulta sin estado (`filter` + `upper`), emitiendo solo palabras nuevas de más de 4 caracteres convertidas a mayúsculas.

In [6]:
# Detener cualquier stream previo activo en Spark
for q in spark.streams.active:
    q.stop()

# Ejercicio 1 (cont.): Modo de salida "append" (solo para transformaciones sin estado)
from pyspark.sql.functions import upper, length

palabrasLargasDF = (
    palabrasDF
    .filter(length("palabra") > 4)
    .select(upper("palabra").alias("palabra_mayuscula"))
)

appendQuery = (
    palabrasLargasDF.writeStream
    .format("console")
    .outputMode("append")
    .start()
)

appendQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------+
|palabra_mayuscula|
+-----------------+
+-----------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------------+
|palabra_mayuscula|
+-----------------+
|           APACHE|
|            SPARK|
|        STREAMING|
+-----------------+


## 4. Ejercicio 2: Triggers (`processingTime="1 minute"`)

Configuramos el trigger para que Spark dispare el procesamiento exactamente cada **1 minuto** (`processingTime="1 minute"`). Al enviar frases espaciadas por menos de un minuto a través de Netcat, Spark las retiene y las procesa juntas cuando se cumple el intervalo de 60 segundos.

In [3]:
# Detener streams previos activos
for q in spark.streams.active:
    q.stop()

# Definir la lectura del socket y transformación de conteo
lineasDF = (
    spark.readStream
    .format("socket")
    .option("host", "localhost")
    .option("port", 9999)
    .load()
)
palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

# Ejercicio 2: Triggers - Intervalo fijo de 1 minuto (processingTime = "1 minute")
wordCountTrigger = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("complete")
    .trigger(processingTime="1 minute")
    .start()
)

wordCountTrigger.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
| prueba|    1|
| minuto|    1|
|trigger|    1|
|     de|    1|
|  spark|    1|
|   hola|    1|
|     un|    1|
+-------+-----+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

## . Cierre Limpio de la Sesión de Spark

In [4]:
# Detener cualquier stream activo y cerrar Spark limpiamente
for q in spark.streams.active:
    q.stop()

spark.stop()
print("Sesión de Spark Streaming finalizada correctamente.")

Sesión de Spark Streaming finalizada correctamente.
